In [2]:
import os
import pandas as pd
from sqlalchemy import Float, Integer, String, create_engine, delete, select
from sqlalchemy.exc import SQLAlchemyError
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column


# Definición del esquema de destino (DDL con SQLAlchemy 2.0)
class Base(DeclarativeBase):
  pass


class ResumenVentasCategoria(Base):
  __tablename__ = "tabla_resumen_ventas_categoria"

  id: Mapped[int] = mapped_column(primary_key=True)
  categoria: Mapped[str] = mapped_column(
      String(100), unique=True, nullable=False
  )
  cantidad_ordenes: Mapped[int] = mapped_column(Integer, nullable=False)
  ingresos_totales_usd: Mapped[float] = mapped_column(Float, nullable=False)
  ticket_promedio_usd: Mapped[float] = mapped_column(Float, nullable=False)


print("Esquema definido correctamente.")

Esquema definido correctamente.


In [3]:
def generar_fuente_datos_ejemplo(ruta_csv: str) -> None:
  """Crea un dataset CSV local simulando datos crudos con nulos, espacios y duplicados."""
  if not os.path.exists(ruta_csv):
    datos_crudos = {
        "id_orden": [101, 102, 103, 104, 105, 106, 107, 107],
        "categoria": [
            " Electrónica ",
            "hogar",
            "ELECTRÓNICA",
            "Ropa",
            "Hogar",
            None,
            "Ropa",
            "Ropa",
        ],
        "monto_usd": [250.0, 80.5, 450.0, 45.0, -15.0, 120.0, 60.0, 60.0],
    }
    pd.DataFrame(datos_crudos).to_csv(ruta_csv, index=False)
    print(f"[INFO] Archivo fuente generado en '{ruta_csv}'.")


def extraer_datos(ruta_csv: str) -> pd.DataFrame:
  """1. EXTRACCIÓN: Lee los datos desde la fuente local."""
  if not os.path.exists(ruta_csv):
    raise FileNotFoundError(
        f"No se encontró el archivo fuente en la ruta: {ruta_csv}"
    )

  df = pd.read_csv(ruta_csv)
  print(f"[1. EXTRACCIÓN] Se leyeron {len(df)} registros crudos.")
  return df


def transformar_datos(df: pd.DataFrame) -> pd.DataFrame:
  """2. TRANSFORMACIÓN: Limpieza de nulos, duplicados, normalización y agregación."""
  df_limpio = df.drop_duplicates(subset=["id_orden"]).copy()
  df_limpio = df_limpio.dropna(subset=["categoria", "monto_usd"])
  df_limpio = df_limpio[df_limpio["monto_usd"] > 0]
  df_limpio["categoria"] = (
      df_limpio["categoria"].str.strip().str.capitalize()
  )

  df_agregado = (
      df_limpio.groupby("categoria", as_index=False)
      .agg(
          cantidad_ordenes=("id_orden", "count"),
          ingresos_totales_usd=("monto_usd", "sum"),
          ticket_promedio_usd=("monto_usd", "mean"),
      )
      .round(2)
  )

  print(
      f"[2. TRANSFORMACIÓN] Datos limpios y agregados en {len(df_agregado)}"
      " categorías."
  )
  return df_agregado


def cargar_datos(df_agregado: pd.DataFrame, url_bd: str) -> None:
  """3. CARGA: Crea el esquema DDL y persiste los datos evitando duplicidad."""
  motor = create_engine(url_bd, echo=False)
  Base.metadata.create_all(motor)

  with Session(motor) as sesion:
    try:
      # Limpieza previa para evitar duplicados si se vuelve a ejecutar (Idempotencia)
      sesion.execute(delete(ResumenVentasCategoria))

      registros = [
          ResumenVentasCategoria(
              categoria=fila["categoria"],
              cantidad_ordenes=int(fila["cantidad_ordenes"]),
              ingresos_totales_usd=float(fila["ingresos_totales_usd"]),
              ticket_promedio_usd=float(fila["ticket_promedio_usd"]),
          )
          for _, fila in df_agregado.iterrows()
      ]

      sesion.add_all(registros)
      sesion.commit()
      print(
          f"[3. CARGA] {len(registros)} registros cargados sin duplicados en la"
          " base de datos."
      )

      print("\n--- Verificación de Datos Persistidos en SQLite ---")
      resultados = sesion.scalars(select(ResumenVentasCategoria)).all()
      for r in resultados:
        print(
            f"ID: {r.id} | Categoría: {r.categoria:<12} | Órdenes:"
            f" {r.cantidad_ordenes} | Total: ${r.ingresos_totales_usd:.2f} |"
            f" Promedio: ${r.ticket_promedio_usd:.2f}"
        )

    except SQLAlchemyError as error_bd:
      sesion.rollback()
      print(f"[ERROR DE BASE DE DATOS] Se ejecutó rollback: {error_bd}")
      raise


print("Funciones ETL cargadas correctamente.")

Funciones ETL cargadas correctamente.


In [4]:
ruta_fuente = "ventas_crudas.csv"
url_base_datos = "sqlite:///ventas_analitica.db"

try:
  generar_fuente_datos_ejemplo(ruta_fuente)
  df_crudo = extraer_datos(ruta_fuente)
  df_transformado = transformar_datos(df_crudo)
  cargar_datos(df_transformado, url_base_datos)
  print("\n[ÉXITO] Pipeline ETL finalizado correctamente.")
except Exception as error:
  print(f"\n[FALLO DEL PIPELINE] El proceso se detuvo: {error}")

[INFO] Archivo fuente generado en 'ventas_crudas.csv'.
[1. EXTRACCIÓN] Se leyeron 8 registros crudos.
[2. TRANSFORMACIÓN] Datos limpios y agregados en 3 categorías.
[3. CARGA] 3 registros cargados sin duplicados en la base de datos.

--- Verificación de Datos Persistidos en SQLite ---
ID: 1 | Categoría: Electrónica  | Órdenes: 2 | Total: $700.00 | Promedio: $350.00
ID: 2 | Categoría: Hogar        | Órdenes: 1 | Total: $80.50 | Promedio: $80.50
ID: 3 | Categoría: Ropa         | Órdenes: 2 | Total: $105.00 | Promedio: $52.50

[ÉXITO] Pipeline ETL finalizado correctamente.
